In [ ]:
#在谷歌colab中运行此cell
!git clone https://github.com/iDPLAB/WHIS_python.git
!pip install torch torchaudio numpy matplotlib
import sys
sys.path.append('/content/WHIS_python')

In [ ]:
# Copyright (c)the Lab of Intelligent Data Processing, Wakayama University.
# All rights reserved.

import numpy as np
import matplotlib.pyplot as plt
import gradio as gr
import time

try:
    from gcfb_v234.gcfb_v234 import gcfb_v234 as GCFBv234
    from gcfb_v234.gcfb_v234 import gcfb_v23_asym_func_in_out
except Exception:
    GCFBv234 = None
    gcfb_v23_asym_func_in_out = None
    print("Warning: gcfb_v234 not available — GCFB init / IO-based HLact will be disabled.")

try:
    from WHISv30_Batch import WHISv30_Batch
    WHIS_BATCH_FUNC = WHISv30_Batch
except Exception:
    try:
        from WHISv30_Batch import WHISv30_batch
        WHIS_BATCH_FUNC = WHISv30_batch
    except Exception:
        WHIS_BATCH_FUNC = None
        print("Warning: WHISv30_Batch not available — run_whis will be a no-op.")


FREQS = np.array([125, 250, 500, 1000, 2000, 4000, 8000])

AUDIOGRAM_DB = {
    "HL1_Example": np.array([10, 4, 10, 13, 48, 58, 79]),
    "HL2_Tsuiki2002_80yr": np.array([23.5, 24.3, 26.8, 27.9, 32.9, 48.3, 68.5]),
    "HL3_ISO7029_70yr_male": np.array([8, 8, 9, 10, 19, 43, 59]),
    "HL4_ISO7029_70yr_female": np.array([8, 8, 9, 10, 16, 24, 41]),
    "HL5_ISO7029_60yr_male": np.array([5, 5, 6, 7, 12, 28, 39]),
    "HL6_ISO7029_60yr_female": np.array([5, 5, 6, 7, 11, 16, 26]),
    "HL7_Example_Otosclerosis": np.array([50, 55, 50, 50, 40, 25, 20]),
    "HL8_Example_NoiseInduced": np.array([15, 10, 15, 10, 10, 40, 20]),
}


class WHISParam:
    def __init__(self, fs=48000):
        self.fs = fs
        self.SPLdB_CalibTone = 80
        self.SrcSndSPLdB_default = 65
        self.SrcSndSPLdB = 65

        self.gcparam = type("obj", (), {})()

        self.hloss = type("obj", (), {})()
        self.hloss.type = "HL1"
        self.hloss_type = "HL1"

        self.calibtone = type("obj", (), {})()
        self.calibtone.SPLdB = 65

        self.srcsnd = type("obj", (), {})()
        self.srcsnd.SPLdB = 65

        self.swplot = 1
        self.hloss.compression_health = 1.0
        self.synth_method = "DTVF"

        self.FaudgramList = FREQS.copy()
        self.DegreeCompression_Faudgram = np.zeros_like(FREQS)
        self.HLdB_LossLinear = np.zeros_like(FREQS)
        self.HearingLevelVal = np.zeros_like(FREQS)


def init_table_degree_compression():
    Table_DegreeCompressionPreSet = np.array([1.0, 2 / 3, 0.5, 1 / 3, 0.0], dtype=float)

    Table_HLdB_DegreeCompressionPreSet = np.array([
        [34.0000, 19.6000,  9.7000,  4.9000,  0.5000,  5.4000,  3.3000],
        [38.8566, 30.6594, 25.6976, 22.1259, 18.0419, 23.8148, 21.4690],
        [40.9698, 34.9736, 32.1693, 29.7996, 26.3673, 32.4441, 30.0962],
        [42.9096, 38.6140, 37.4731, 36.3657, 33.5918, 39.5464, 37.7582],
        [46.3609, 44.4196, 45.0619, 45.1542, 43.6208, 48.4398, 47.2515],
    ], dtype=float)

    Table_HLdB_DegreeCompressionPreSet = (
        Table_HLdB_DegreeCompressionPreSet - Table_HLdB_DegreeCompressionPreSet[0, :]
    )

    return Table_DegreeCompressionPreSet, Table_HLdB_DegreeCompressionPreSet


def create_whisparam(fs, audiogram_type, compression_health):
    try:
        import Param as _Param_module
        whisparam = _Param_module.WHISparam()
    except Exception:
        whisparam = WHISParam(fs)

    whisparam.fs = fs

    whisparam.hloss.type = audiogram_type[:3]
    whisparam.hloss_type = audiogram_type[:3]

    whisparam.calibtone.SPLdB = 65
    whisparam.srcsnd.SPLdB = 65

    whisparam.gcparam.out_mid_crct = "FreeField"

    whisparam.swplot = 1
    whisparam.hloss.compression_health = compression_health
    whisparam.synth_method = "DTVF"

    whisparam.FaudgramList = FREQS.copy()
    whisparam.HearingLevelList = np.vstack(list(AUDIOGRAM_DB.values()))

    whisparam.Table_getComp = np.array([100, 67, 50, 33, 0])
    whisparam.Table_DegreeCompressionPreSet, whisparam.Table_HLdB_DegreeCompressionPreSet = (
        init_table_degree_compression()
    )

    whisparam.Table_DegreeCompressionPreSet = np.asarray(
        whisparam.Table_DegreeCompressionPreSet, dtype=float
    )
    whisparam.Table_HLdB_DegreeCompressionPreSet = np.asarray(
        whisparam.Table_HLdB_DegreeCompressionPreSet, dtype=float
    )

    whisparam.HearingLevelVal = AUDIOGRAM_DB[audiogram_type].astype(float).copy()
    whisparam.HLdB_LossCompression = whisparam.HearingLevelVal.copy()

    return whisparam


def update_whisparam_from_gui(whisparam, audiogram_type, compression_percent):
    if whisparam is None:
        return None

    whisparam.hloss.compression_health = compression_percent

    whisparam.hloss.type = audiogram_type[:3]
    whisparam.hloss_type = audiogram_type[:3]

    return whisparam


def interp1_linear_extrap(x, y, xq):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if np.any(np.diff(x) < 0):
        sort_idx = np.argsort(x)
        x = x[sort_idx]
        y = y[sort_idx]

    return np.interp(
        xq,
        x,
        y,
        left=y[0] + (xq - x[0]) * (y[1] - y[0]) / (x[1] - x[0]) if len(x) > 1 else y[0],
        right=y[-1] + (xq - x[-1]) * (y[-1] - y[-2]) / (x[-1] - x[-2]) if len(x) > 1 else y[-1],
    )


def WHIS_CalDegreeCmprsLin_py(ParamHI):
    LenFag = len(ParamHI.FaudgramList)
    Table_HLdB = np.asarray(ParamHI.Table_HLdB_DegreeCompressionPreSet, dtype=float)
    Table_Deg = np.asarray(ParamHI.Table_DegreeCompressionPreSet, dtype=float)

    HLdB_LossCompression = np.asarray(ParamHI.HLdB_LossCompression, dtype=float).copy()
    DCcmpnst = np.zeros(LenFag, dtype=float)

    for nfag in range(LenFag):
        DC = interp1_linear_extrap(
            Table_HLdB[:, nfag],
            Table_Deg,
            HLdB_LossCompression[nfag],
        )

        DC = max(min(DC, 1.0), 0.0)
        DCcmpnst[nfag] = DC

        HLdB_LossCompression[nfag] = interp1_linear_extrap(
            Table_Deg,
            Table_HLdB[:, nfag],
            DC,
        )

    ParamHI.DCcmpnst = DCcmpnst
    ParamHI.DegreeCompression_Faudgram = DCcmpnst

    ParamHI.HLdB_LossCompression = HLdB_LossCompression - Table_HLdB[0, :]

    ParamHI.HLdB_LossLinear = (
        np.asarray(ParamHI.HearingLevelVal, dtype=float) - ParamHI.HLdB_LossCompression
    )

    return ParamHI


def compute_HLact_gcfb(hl_total, compression_health, gc_param, gc_resp, fr1query, pref_db=50.0):
    if gcfb_v23_asym_func_in_out is None:
        return np.asarray(hl_total, dtype=float).copy()

    hl_act = np.zeros_like(hl_total, dtype=float)
    pin_ref = np.array([pref_db])

    for i in range(len(hl_total)):
        _, io_nh, _ = gcfb_v23_asym_func_in_out(
            gc_param,
            gc_resp,
            fr1query[i],
            compression_health=1.0,
            pin_db=pin_ref,
        )

        target_out = io_nh[0]

        pin_grid = np.linspace(pref_db - hl_total[i] - 40, pref_db + 10, 400)

        _, io_hl_curve, _ = gcfb_v23_asym_func_in_out(
            gc_param,
            gc_resp,
            fr1query[i],
            compression_health=compression_health[i],
            pin_db=pin_grid,
        )

        idx = np.argmin(np.abs(io_hl_curve - target_out))
        pin_hl = pin_grid[idx]

        hl_act[i] = pref_db - pin_hl
        hl_act[i] = max(hl_act[i], hl_total[i])

    return hl_act


def plot_audiogram(audiogram_type, compression_percent, gc_param, gc_resp, fr1query):
    if audiogram_type is None:
        return None, None, None

    hl_total = AUDIOGRAM_DB[audiogram_type].astype(float)

    N = hl_total.size
    xii = np.arange(1, N + 1)

    whisparam = create_whisparam(
        fs=48000,
        audiogram_type=audiogram_type,
        compression_health=(compression_percent / 100.0),
    )

    whisparam.HearingLevelVal = hl_total.copy()

    table_get_comp = getattr(whisparam, "Table_getComp", np.array([100, 67, 50, 33, 0]))

    numcomp_idx = int(compression_percent)
    compression_percent = TABLE_GET_COMP[numcomp_idx]

    numcomp_idx = max(0, min(numcomp_idx, len(table_get_comp) - 1))

    table_hl = np.asarray(whisparam.Table_HLdB_DegreeCompressionPreSet, dtype=float)

    preset_row = table_hl[numcomp_idx, :]

    whisparam.HLdB_LossCompression = np.minimum(
        whisparam.HearingLevelVal,
        preset_row,
    )

    whisparam = WHIS_CalDegreeCmprsLin_py(whisparam)

    hl_act = np.asarray(whisparam.HLdB_LossCompression, dtype=float)

    fig, ax = plt.subplots(figsize=(9, 7))
    ax.cla()

    ax.plot([-1, N + 1], [0, 0], "k", linewidth=1.5)
    ax.set_xlim(0, N + 1)

    ax.plot(
        xii,
        whisparam.HearingLevelVal,
        "ko-",
        markersize=9,
        linewidth=1.5,
        label="HL_total",
    )

    nTxt = 6 if N >= 6 else N
    idx_for_text = min(nTxt - 1, N - 1)

    ax.text(
        nTxt + 0.1,
        whisparam.HearingLevelVal[idx_for_text] - 2,
        r"HL$_{total}$",
        fontsize=10,
    )

    hl_normal = table_hl[0, :]

    ax.plot(xii, hl_normal, "gx-", linewidth=1.5, label="Normal Hearing")
    ax.text(
        nTxt,
        hl_normal[idx_for_text] - 5,
        "Normal Hearing",
        fontsize=10,
    )

    ax.plot(
        xii,
        preset_row,
        "cs:",
        label="Preset (raw table)",
    )

    ax.plot(
        xii,
        hl_act,
        "mx-",
        markersize=9,
        linewidth=1.2,
        label="HL_act",
    )

    ax.text(
        nTxt + 0.1,
        hl_act[idx_for_text] + 4,
        r"HL$_{act}$",
        fontsize=10,
    )

    ax.set_xticks(xii)
    ax.set_xticklabels([str(int(f)) for f in FREQS])

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Hearing Level (dB)")

    ax.invert_yaxis()
    ax.set_ylim(120, -20)

    ax.grid(True)
    ax.legend()

    hl_table_vals = whisparam.HearingLevelVal.reshape(1, -1)
    hl_table_out = np.array([[f"{v:.2f}" for v in hl_table_vals[0]]])

    comp_vals = (whisparam.DCcmpnst * 100.0).reshape(1, -1)
    comp_table_out = np.array([[f"{v:.2f}" for v in comp_vals[0]]])

    return fig, hl_table_out, comp_table_out


def load_audio_and_init(audio, audiogram_type, compression_percent):
    if audio is None:
        return None, None, None, None

    fs, snd = audio

    compression_percent = TABLE_GET_COMP[int(compression_percent)]
    compression_health = compression_percent / 100.0

    whisparam = create_whisparam(
        fs,
        audiogram_type,
        compression_health,
    )

    gcparam = whisparam.gcparam

    try:
        gcparam.ctrl = "dynamic"
        gcparam.dyn_hpaf_str_prc = "frame-base"
        gcparam.hloss = whisparam.hloss
        gcparam.hloss_type = whisparam.hloss_type
        gcparam.hloss_compression_health = whisparam.hloss.compression_health
    except Exception:
        pass

    if GCFBv234 is not None:
        try:
            Snd4GCFB = snd

            _, _, GCparamHL, GCrespHL = GCFBv234(Snd4GCFB, gcparam)

            fr1 = GCparamHL.fr1 if hasattr(GCparamHL, "fr1") else whisparam.FaudgramList

            return GCparamHL, GCrespHL, fr1, whisparam

        except Exception as e:
            print("Warning: GCFBv234 call failed:", e)
            return None, None, whisparam.FaudgramList, whisparam

    else:
        return None, None, whisparam.FaudgramList, whisparam


def to_int16_preserve_relative(src, whis):
    """
    Convert SrcSnd and SndWHIS to int16 with the same scale.

    Important:
    - Do NOT normalize src and whis separately.
    - If both signals are already within [-1, 1], use scale = 1.0.
      This preserves the original digital amplitude.
    - If either signal exceeds [-1, 1], use a common scale to avoid clipping.
      This still preserves the relative level difference.
    """

    src = np.asarray(src, dtype=np.float64)
    whis = np.asarray(whis, dtype=np.float64)

    max_abs_src = np.max(np.abs(src)) if src.size > 0 else 0.0
    max_abs_whis = np.max(np.abs(whis)) if whis.size > 0 else 0.0

    max_abs = max(max_abs_src, max_abs_whis, 1e-12)

    if max_abs <= 1.0:
        scale = 1.0
    else:
        scale = max_abs
        print(
            "[Warning] Audio amplitude exceeded [-1, 1]. "
            "A common scale was applied to avoid clipping:",
            scale,
        )

    src_i16 = (np.clip(src / scale, -1.0, 1.0) * 32767.0).astype(np.int16)
    whis_i16 = (np.clip(whis / scale, -1.0, 1.0) * 32767.0).astype(np.int16)

    print("Output int16 common scale:", scale)

    return src_i16, whis_i16


def rms_db(x):
    x = np.asarray(x, dtype=float)
    return 20 * np.log10(np.sqrt(np.mean(x ** 2)) + 1e-12)


def peak_db(x):
    x = np.asarray(x, dtype=float)
    return 20 * np.log10(np.max(np.abs(x)) + 1e-12)


def run_whis(audio, whisparam, audiogram_type, compression_percent):
    if audio is None or whisparam is None:
        return None, None

    if WHIS_BATCH_FUNC is None:
        print("WHISv30Batch not available.")
        return None, None

    compression_percent = TABLE_GET_COMP[int(compression_percent)]

    fs, snd = audio

    whisparam = update_whisparam_from_gui(
        whisparam,
        audiogram_type,
        compression_percent,
    )

    SndWHIS, SrcSnd, _, whisparam_out = WHIS_BATCH_FUNC(snd, whisparam)

    
    print("========== Level before Gradio output ==========")
    print("SrcSnd RMS dB:", rms_db(SrcSnd))
    print("SndWHIS RMS dB:", rms_db(SndWHIS))
    print("SrcSnd Peak dB:", peak_db(SrcSnd))
    print("SndWHIS Peak dB:", peak_db(SndWHIS))
    print("RMS diff WHIS-Src:", rms_db(SndWHIS) - rms_db(SrcSnd), "dB")
    print("Peak diff WHIS-Src:", peak_db(SndWHIS) - peak_db(SrcSnd), "dB")
    print("===============================================")

    SrcSnd_i16, SndWHIS_i16 = to_int16_preserve_relative(SrcSnd, SndWHIS)

    return (fs, SrcSnd_i16), (fs, SndWHIS_i16)


# ===============================
# GUI
# ===============================
with gr.Blocks(title="WHIS v30 GUI") as demo:

    gc_param_state = gr.State(None)
    gc_resp_state = gr.State(None)
    fr1query_state = gr.State(None)
    whisparam_state = gr.State(None)

    gr.Markdown("WHIS")

    with gr.Row():
        with gr.Column(scale=3):
            audiogram_plot = gr.Plot()
            hl_table = gr.Dataframe(headers=[str(f) for f in FREQS], row_count=1)
            comp_table = gr.Dataframe(headers=[str(f) for f in FREQS], row_count=1)

        with gr.Column(scale=1):
            audiogram_sel = gr.Dropdown(
                list(AUDIOGRAM_DB.keys()),
                value="HL1_Example",
            )

            TABLE_GET_COMP = np.array([100, 67, 50, 33, 0])

            compression_slider = gr.Slider(
                minimum=0,
                maximum=4,
                step=1,
                value=2,
                label="Compression Level (0=100%, 4=0%)",
            )

            audio_upload = gr.Audio(
                type="numpy",
                label="Load Source Sound",
            )

            run_btn = gr.Button("▶ Start Processing")

            audio_out_src = gr.Audio(
                type="numpy",
                label="Source Sound (SrcSnd)",
            )

            audio_out_whis = gr.Audio(
                type="numpy",
                label="WHIS Output (SndWHIS)",
            )

    audio_upload.change(
        load_audio_and_init,
        [audio_upload, audiogram_sel, compression_slider],
        [gc_param_state, gc_resp_state, fr1query_state, whisparam_state],
    )

    audiogram_sel.change(
        plot_audiogram,
        [
            audiogram_sel,
            compression_slider,
            gc_param_state,
            gc_resp_state,
            fr1query_state,
        ],
        [audiogram_plot, hl_table, comp_table],
    )

    compression_slider.change(
        plot_audiogram,
        [
            audiogram_sel,
            compression_slider,
            gc_param_state,
            gc_resp_state,
            fr1query_state,
        ],
        [audiogram_plot, hl_table, comp_table],
    )

    run_btn.click(
        run_whis,
        [audio_upload, whisparam_state, audiogram_sel, compression_slider],
        [audio_out_src, audio_out_whis],
    )

demo.launch()